# Drift Calibration — CGCS / Triplet-Phase-Lock Starter Notebook

**Repository:** `quantum-hardware-company`  
**Directory:** `calibration/drift/`

This notebook initializes a drift calibration workflow:

1. Generate synthetic calibration data across repeated time blocks.
2. Simulate slow drift in Rabi frequency and readout offset.
3. Fit each time block independently.
4. Track fitted parameters over lab time.
5. Analyze drift residuals and stability.
6. Compute CGCS / Triplet-Phase-Lock stability metrics over time.
7. Save figures and a machine-readable JSON summary.

## Principle

Drift calibration measures whether hardware behavior remains stable across repeated calibrations.

Rabi calibrates amplitude.  
Ramsey calibrates phase/coherence.  
RB checks gate-survival decay.  
Drift checks whether those calibrations continue to hold over time.


## Drift model

For each calibration block, we use a Rabi-like response:

$$
P(t)=A\sin^2\left(\frac{\Omega t}{2}\right)+B
$$

but allow parameters to drift across lab time:

$$
\Omega_k = \Omega_0 + \delta\Omega(k)
$$

$$
B_k = B_0 + \delta B(k)
$$

where $k$ indexes repeated calibration blocks.

This starter notebook uses synthetic data and is meant to establish the workflow before replacing data with experiment outputs.


In [ ]:
import os
import json

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

np.random.seed(9423)

FIG_DIR = "figures"
RESULTS_DIR = "results"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


In [ ]:
def rabi_model(t, A, Omega, B):
    """Rabi oscillation probability model."""
    return A * np.sin(0.5 * Omega * t) ** 2 + B


def fit_model(model, x, y, p0, bounds=(-np.inf, np.inf)):
    """Fit model to data and return best-fit params + covariance."""
    params, cov = curve_fit(model, x, y, p0=p0, bounds=bounds, maxfev=30000)
    return params, cov


def residuals(model, x, y, params):
    """Observed minus fitted values."""
    return y - model(x, *params)


def phase_lock_metric(signal, fit):
    """Cosine similarity between observed signal and fitted model."""
    dot = np.dot(signal, fit)
    norm = np.linalg.norm(signal) * np.linalg.norm(fit)
    if norm == 0:
        return np.nan
    return dot / norm


def is_phase_locked(metric, threshold=1 / np.sqrt(2)):
    """CGCS threshold: cos(theta) >= 1/sqrt(2), equivalent to <= 45 degrees."""
    return metric >= threshold


def autocorrelation(x):
    """Simple non-normalized autocorrelation for residual structure checks."""
    x = x - np.mean(x)
    result = np.correlate(x, x, mode="full")
    return result[result.size // 2:]


## Generate synthetic drift calibration data

Each block represents one calibration run at a later lab time.

The synthetic system includes:

- slow sinusoidal Rabi-frequency drift,
- slow monotonic readout-offset drift,
- measurement noise,
- bounded probability readout.


In [ ]:
# Repeated calibration blocks
n_blocks = 28
block_times = np.arange(n_blocks)

# Pulse duration axis inside each calibration block
n_points_per_block = 120
pulse_t = np.linspace(0, 10, n_points_per_block)

# Base parameters
base_A = 0.88
base_Omega = 2.45
base_B = 0.06

# Synthetic parameter drift across calibration blocks
true_A_blocks = base_A + 0.015 * np.sin(2 * np.pi * block_times / 17)
true_Omega_blocks = base_Omega + 0.055 * np.sin(2 * np.pi * block_times / 19 + 0.4)
true_B_blocks = base_B + 0.0018 * block_times + 0.006 * np.sin(2 * np.pi * block_times / 13)

# Store synthetic observations
Y_obs = []
Y_clean = []

for k in range(n_blocks):
    y_clean = rabi_model(
        pulse_t,
        true_A_blocks[k],
        true_Omega_blocks[k],
        true_B_blocks[k],
    )
    measurement_noise = 0.04 * np.random.randn(n_points_per_block)
    y_obs = y_clean + measurement_noise
    y_obs = np.clip(y_obs, 0, 1)

    Y_clean.append(y_clean)
    Y_obs.append(y_obs)

Y_clean = np.array(Y_clean)
Y_obs = np.array(Y_obs)

print(f"Generated {n_blocks} calibration blocks.")
print(f"Pulse samples per block: {n_points_per_block}")
print(f"Omega drift range: {true_Omega_blocks.min():.4f} to {true_Omega_blocks.max():.4f}")
print(f"Offset drift range: {true_B_blocks.min():.4f} to {true_B_blocks.max():.4f}")


## Plot synthetic calibration blocks

This shows how repeated calibration traces shift over lab time.


In [ ]:
plt.figure(figsize=(9, 5))
for k in range(n_blocks):
    if k % 4 == 0:
        plt.plot(pulse_t, Y_obs[k], alpha=0.75, label=f"block {k}")

plt.xlabel("time / pulse duration")
plt.ylabel("excited-state probability")
plt.title("Drift calibration: repeated Rabi-like calibration blocks")
plt.legend(ncol=2)
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/drift_01_repeated_blocks.png", dpi=300)
plt.savefig(f"{FIG_DIR}/drift_01_repeated_blocks.pdf")

plt.show()


## Fit each calibration block

Each block is fit independently to recover time-dependent calibration parameters.


In [ ]:
fit_params = []
fit_stds = []
fit_curves = []
block_rmse = []
block_phase_lock = []
block_angle_deg = []

initial_guess = [0.85, 2.40, 0.05]
lower_bounds = [0.0, 0.0, 0.0]
upper_bounds = [1.2, 5.0, 0.3]

threshold = 1 / np.sqrt(2)

for k in range(n_blocks):
    params, cov = fit_model(
        rabi_model,
        pulse_t,
        Y_obs[k],
        p0=initial_guess,
        bounds=(lower_bounds, upper_bounds),
    )

    y_fit = rabi_model(pulse_t, *params)
    res = Y_obs[k] - y_fit
    rmse = np.sqrt(np.mean(res**2))
    metric = phase_lock_metric(Y_obs[k], y_fit)
    angle = np.degrees(np.arccos(np.clip(metric, -1, 1)))

    fit_params.append(params)
    fit_stds.append(np.sqrt(np.diag(cov)))
    fit_curves.append(y_fit)
    block_rmse.append(rmse)
    block_phase_lock.append(metric)
    block_angle_deg.append(angle)

fit_params = np.array(fit_params)
fit_stds = np.array(fit_stds)
fit_curves = np.array(fit_curves)
block_rmse = np.array(block_rmse)
block_phase_lock = np.array(block_phase_lock)
block_angle_deg = np.array(block_angle_deg)

A_fit = fit_params[:, 0]
Omega_fit = fit_params[:, 1]
B_fit = fit_params[:, 2]

print("Completed block-wise fits.")
print(f"Mean RMSE: {block_rmse.mean():.6f}")
print(f"Mean CGCS phase-lock metric: {block_phase_lock.mean():.6f}")
print(f"Max CGCS angle: {block_angle_deg.max():.3f} degrees")


## Parameter drift tracking

This is the core drift-calibration output: fitted parameters as functions of lab time.


In [ ]:
plt.figure(figsize=(9, 5))
plt.errorbar(block_times, Omega_fit, yerr=fit_stds[:, 1], fmt="o-", capsize=3, label="fitted Omega")
plt.plot(block_times, true_Omega_blocks, linestyle="--", label="true synthetic Omega")
plt.xlabel("calibration block / lab time index")
plt.ylabel("Rabi frequency Omega")
plt.title("Drift tracking: Rabi frequency over repeated calibration blocks")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/drift_02_omega_tracking.png", dpi=300)
plt.savefig(f"{FIG_DIR}/drift_02_omega_tracking.pdf")

plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
plt.errorbar(block_times, B_fit, yerr=fit_stds[:, 2], fmt="o-", capsize=3, label="fitted offset B")
plt.plot(block_times, true_B_blocks, linestyle="--", label="true synthetic offset B")
plt.xlabel("calibration block / lab time index")
plt.ylabel("readout offset B")
plt.title("Drift tracking: readout offset over repeated calibration blocks")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/drift_03_offset_tracking.png", dpi=300)
plt.savefig(f"{FIG_DIR}/drift_03_offset_tracking.pdf")

plt.show()


## Drift residuals

After fitting each block, compare recovered parameters to the known synthetic drift.

For real data, this section becomes comparison against smoothed estimates, previous calibration state, or a drift model.


In [ ]:
Omega_error = Omega_fit - true_Omega_blocks
B_error = B_fit - true_B_blocks

plt.figure(figsize=(9, 4))
plt.axhline(0, linewidth=1)
plt.plot(block_times, Omega_error, marker="o", linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("fitted Omega - true Omega")
plt.title("Drift residuals: frequency tracking error")
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/drift_04_omega_error.png", dpi=300)
plt.savefig(f"{FIG_DIR}/drift_04_omega_error.pdf")

plt.show()


In [ ]:
plt.figure(figsize=(9, 4))
plt.axhline(0, linewidth=1)
plt.plot(block_times, B_error, marker="o", linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("fitted B - true B")
plt.title("Drift residuals: offset tracking error")
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/drift_05_offset_error.png", dpi=300)
plt.savefig(f"{FIG_DIR}/drift_05_offset_error.pdf")

plt.show()


## Stability diagnostics

Track fit quality and CGCS phase-lock across calibration blocks.


In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(block_times, block_rmse, marker="o", linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("fit RMSE")
plt.title("Drift calibration: fit RMSE over time")
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/drift_06_rmse_over_time.png", dpi=300)
plt.savefig(f"{FIG_DIR}/drift_06_rmse_over_time.pdf")

plt.show()


In [ ]:
plt.figure(figsize=(9, 4))
plt.axhline(threshold, linewidth=1, linestyle="--", label="45° threshold")
plt.plot(block_times, block_phase_lock, marker="o", linewidth=1, label="block phase-lock metric")
plt.ylim(0.65, 1.02)
plt.xlabel("calibration block / lab time index")
plt.ylabel("cosine similarity")
plt.title("CGCS phase-lock stability over calibration blocks")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/drift_07_phase_lock_over_time.png", dpi=300)
plt.savefig(f"{FIG_DIR}/drift_07_phase_lock_over_time.pdf")

plt.show()


## CGCS stability summary

A stable drift-calibration run should remain above the 45° threshold.

Here, phase-lock is evaluated block by block rather than only once at the end.


In [ ]:
min_metric = float(np.min(block_phase_lock))
mean_metric = float(np.mean(block_phase_lock))
max_angle = float(np.max(block_angle_deg))
all_locked = bool(np.all(block_phase_lock >= threshold))

print(f"Minimum CGCS phase-lock metric = {min_metric:.6f}")
print(f"Mean CGCS phase-lock metric    = {mean_metric:.6f}")
print(f"Maximum angle                  = {max_angle:.3f} degrees")
print(f"All blocks phase-locked?       = {all_locked}")


## Parameter autocorrelation

Parameter autocorrelation checks whether fitted drift contains time-correlated structure.

This is expected for slow hardware drift and distinguishes drift from independent fit noise.


In [ ]:
Omega_ac = autocorrelation(Omega_fit)
B_ac = autocorrelation(B_fit)

plt.figure(figsize=(8, 4))
plt.axhline(0, linewidth=1)
plt.plot(Omega_ac[:14], marker="o", linewidth=1, label="Omega autocorrelation")
plt.plot(B_ac[:14], marker="o", linewidth=1, label="B autocorrelation")
plt.xlabel("lag")
plt.ylabel("correlation")
plt.title("Parameter autocorrelation: drift structure check")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/drift_08_parameter_autocorr.png", dpi=300)
plt.savefig(f"{FIG_DIR}/drift_08_parameter_autocorr.pdf")

plt.show()


## Calibration summary


In [ ]:
summary = {
    "n_blocks": int(n_blocks),
    "n_points_per_block": int(n_points_per_block),
    "Omega_fit_mean": float(np.mean(Omega_fit)),
    "Omega_fit_std_over_blocks": float(np.std(Omega_fit)),
    "Omega_fit_min": float(np.min(Omega_fit)),
    "Omega_fit_max": float(np.max(Omega_fit)),
    "B_fit_mean": float(np.mean(B_fit)),
    "B_fit_std_over_blocks": float(np.std(B_fit)),
    "B_fit_min": float(np.min(B_fit)),
    "B_fit_max": float(np.max(B_fit)),
    "mean_block_rmse": float(np.mean(block_rmse)),
    "max_block_rmse": float(np.max(block_rmse)),
    "min_phase_lock_metric": float(min_metric),
    "mean_phase_lock_metric": float(mean_metric),
    "phase_lock_threshold": float(threshold),
    "max_phase_lock_angle_degrees": float(max_angle),
    "all_blocks_phase_locked": bool(all_locked),
}

summary


## Save calibration summary


In [ ]:
with open(f"{RESULTS_DIR}/drift_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved {RESULTS_DIR}/drift_summary.json")


## Zip outputs for download from Colab

Run this cell after all figures/results have been generated.


In [ ]:
!zip -r drift_outputs.zip figures results


## Calibration layer status

The calibration layer now has four linked workflows:

- `calibration/rabi/` → amplitude / drive response
- `calibration/ramsey/` → phase / detuning / dephasing
- `calibration/rb/` → randomized gate-survival decay
- `calibration/drift/` → stability over repeated calibration blocks

Together, these notebooks form the first measurable system layer for `quantum-hardware-company`.

## Next steps

From here, the repo can move into:

- `noise-mitigation/` for structured drift/noise modeling,
- `control-stack/` for closed-loop calibration updates,
- `control-electronics/` for timing/filtering constraints.

Guiding rule:

> Start where physics shows up.
